In [ ]:
# Основа для модели нейронной сети
from tensorflow.keras.models import Model

# Стандартные слои keras
from tensorflow.keras.layers import Input, Conv2DTranspose, concatenate, Activation, MaxPooling2D, Conv2D, BatchNormalization

# Оптимизатор Adam
from tensorflow.keras.optimizers import Adam

# Дополнительные утилиты keras
from tensorflow.keras import utils

# Инструменты для построения графиков
import matplotlib.pyplot as plt

# Инструменты для работы с изображениями
from tensorflow.keras.preprocessing import image

# Инструменты для работы с массивами
import numpy as np

# Системные инструменты
import time, random, gdown, os

# Дополнительные инструменты для работы с изображениями
from PIL import Image

# Дополнительные инструменты визуализации
import seaborn as sns
sns.set_style('darkgrid')

In [ ]:
# Загрузка датасета из облака (База "Стройка", 256x192)
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l14/construction_256x192.zip', None, quiet=False)

# Для работы с базой "Самолеты" (Практика 3) раскомментируйте строку ниже:
# gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l14/airplane_456x256.zip', None, quiet=True)

# Распаковка скачанного архива
!unzip -qo construction_256x192.zip
# !unzip -qo airplane_456x256.zip

Downloading...
From: https://storage.yandexcloud.net/aiueducation/Content/base/l14/construction_256x192.zip
To: /content/construction_256x192.zip
100%|██████████| 214M/214M [00:14<00:00, 14.5MB/s]


In [ ]:
# Глобальные параметры для базы "Стройка"
IMG_WIDTH = 192               # Ширина картинки
IMG_HEIGHT = 256              # Высота картинки
CLASS_COUNT = 16              # Количество классов на изображении (16 для стройки)
TRAIN_DIRECTORY = 'train'     # Папка с обучающей выборкой
VAL_DIRECTORY = 'val'         # Папка с проверочной выборкой

# Примечание: Для базы "Самолеты" установите значения:
# IMG_WIDTH = 256, IMG_HEIGHT = 456, CLASS_COUNT = 2

In [ ]:
def load_imageset(folder, subset, title):
    # Cписок для хранения изображений выборки
    image_list = []
    cur_time = time.time()

    # Для всех файлов в каталоге по указанному пути:
    for filename in sorted(os.listdir(f'{folder}/{subset}')):
        # Чтение очередной картинки и добавление ее в список
        image_list.append(image.load_img(os.path.join(f'{folder}/{subset}', filename),
                                         target_size=(IMG_WIDTH, IMG_HEIGHT)))

    print('{} выборка загружена. Время загрузки: {:.2f} с'.format(title, time.time() - cur_time))
    print('Количество изображений:', len(image_list))
    return image_list

# Непосредственная загрузка оригинальных картинок и их масок сегментации
train_images = load_imageset(TRAIN_DIRECTORY, 'original', 'Обучающая оригинальная')
val_images = load_imageset(VAL_DIRECTORY, 'original', 'Проверочная оригинальная')

train_segments = load_imageset(TRAIN_DIRECTORY, 'segment', 'Обучающая маски')
val_segments = load_imageset(VAL_DIRECTORY, 'segment', 'Проверочная маски')

Обучающая оригинальная выборка загружена. Время загрузки: 0.17 с
Количество изображений: 1900
Проверочная оригинальная выборка загружена. Время загрузки: 0.01 с
Количество изображений: 100
Обучающая маски выборка загружена. Время загрузки: 0.16 с
Количество изображений: 1900
Проверочная маски выборка загружена. Время загрузки: 0.01 с
Количество изображений: 100


In [ ]:
# Цвета пикселов сегментированных изображений базы "Стройка"
FLOOR = (100, 100, 100)         # Пол (серый)
CEILING = (0, 0, 100)           # Потолок (синий)
WALL = (0, 100, 0)              # Стена (зеленый)
COLUMN = (100, 0, 0)            # Колонна (красный)
APERTURE = (0, 100, 100)        # Проем (темно-бирюзовый)
DOOR = (100, 0, 100)            # Дверь (бордовый)
WINDOW = (100, 100, 0)          # Окно (золотой)
EXTERNAL = (200, 200, 200)      # Внешний мир (светло-серый)
RAILINGS = (0, 200, 0)          # Перила (светло-зеленый)
BATTERY = (200, 0, 0)           # Батареи (светло-красный)
PEOPLE = (0, 200, 200)          # Люди (бирюзовый)
LADDER = (0, 0, 200)            # Лестница (светло-синий)
INVENTORY = (200, 0, 200)       # Инвентарь (розовый)
LAMP = (200, 200, 0)            # Лампа (желтый)
WIRE = (0, 100, 200)            # Провод (голубой)
BEAM = (100, 0, 200)            # Балка (фиолетовый)

CLASS_LABELS = (FLOOR, CEILING, WALL, COLUMN, APERTURE, DOOR, WINDOW, EXTERNAL,
                RAILINGS, BATTERY, PEOPLE, LADDER, INVENTORY, LAMP, WIRE, BEAM)

In [ ]:
# ==============================================================================
# БЛОК ОПРЕДЕЛЕНИЯ КЛАССОВ И КОНВЕРТАЦИИ МАСОК (ВКЛЮЧАЯ rgb_to_5_labels)
# ==============================================================================

# 1. Оригинальная палитра цветов (16 исходных классов из датасета)
FLOOR = (100, 100, 100); CEILING = (0, 0, 100); WALL = (0, 100, 0); COLUMN = (100, 0, 0)
APERTURE = (0, 100, 100); DOOR = (100, 0, 100); WINDOW = (100, 100, 0); EXTERNAL = (200, 200, 200)
RAILINGS = (0, 200, 0); BATTERY = (200, 0, 0); PEOPLE = (0, 200, 200); LADDER = (0, 0, 200)
INVENTORY = (200, 0, 200); LAMP = (200, 200, 0); WIRE = (0, 100, 200); BEAM = (100, 0, 200)

# 2. Группировка 16 исходных классов в 5 новых макро-категорий по условию задачи:
class_0 = [WALL, COLUMN, BEAM]                             # Несущие элементы
class_1 = [FLOOR, CEILING]                                 # Перекрытия
class_2 = [APERTURE, DOOR, WINDOW]                         # Проемы и окна
class_3 = [RAILINGS, BATTERY, LADDER, INVENTORY, LAMP, WIRE] # Коммуникации и объекты
class_4 = [EXTERNAL, PEOPLE]                               # Внешний мир и люди

# 3. Функция преобразования трехканального RGB-изображения в одноканальные метки (0..4)
def rgb_to_5_labels(image_list):
    result = []
    for img in image_list:
        sample = np.array(img)
        # Создаем пустую одноканальную матрицу под размеры картинки
        y = np.zeros((IMG_WIDTH, IMG_HEIGHT, 1), dtype='uint8')

        # Перебираем списки цветов и заменяем их на новые индексы от 0 до 4
        for cl in class_0: y[np.where(np.all(sample == cl, axis=-1))] = 0
        for cl in class_1: y[np.where(np.all(sample == cl, axis=-1))] = 1
        for cl in class_2: y[np.where(np.all(sample == cl, axis=-1))] = 2
        for cl in class_3: y[np.where(np.all(sample == cl, axis=-1))] = 3
        for cl in class_4: y[np.where(np.all(sample == cl, axis=-1))] = 4
        result.append(y)
    return np.array(result)

In [ ]:
# ==============================================================================
# ПОДГОТОВКА И НОРМАЛИЗАЦИЯ ТЕНЗОРОВ (БЕЗ ПЕРЕПОЛНЕНИЯ ПАМЯТИ)
# ==============================================================================

print("Начало подготовки тензоров...")

# Переводим оригинальные изображения в numpy-массивы и нормализуем (делением на 255.0)
x_train = np.array([image.img_to_array(img) for img in train_images]) / 255.0
x_val = np.array([image.img_to_array(img) for img in val_images]) / 255.0

# Конвертируем маски в 5 разреженных классов с помощью созданной функции
y_train = rgb_to_5_labels(train_segments)
y_val = rgb_to_5_labels(val_segments)

print("Данные успешно подготовлены!")
print("Размерность x_train (картинки):", x_train.shape)
print("Размерность y_train (маски 5 классов):", y_train.shape)

Начало подготовки тензоров...
Данные успешно подготовлены!
Размерность x_train (картинки): (1900, 192, 256, 3)
Размерность y_train (маски 5 классов): (1900, 192, 256, 1)


In [ ]:
def unet_model(class_count, input_shape):
    inputs = Input(input_shape)

    # Блок 1 (Энкодер - сжатие)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
    conv1 = Conv2D(64, 3, activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D(pool_size=(2, 2))(conv1)

    # Блок 2
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(pool1)
    conv2 = Conv2D(128, 3, activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D(pool_size=(2, 2))(conv2)

    # Блок 3 (Узкое горлышко / Боттлнек)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(pool2)
    conv3 = Conv2D(256, 3, activation='relu', padding='same')(conv3)

    # Блок 4 (Декодер - восстановление с добавлением Skip Connections)
    up4 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv3)
    merge4 = concatenate([conv2, up4], axis=3)
    conv4 = Conv2D(128, 3, activation='relu', padding='same')(merge4)
    conv4 = Conv2D(128, 3, activation='relu', padding='same')(conv4)

    # Блок 5
    up5 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv4)
    merge5 = concatenate([conv1, up5], axis=3)
    conv5 = Conv2D(64, 3, activation='relu', padding='same')(merge5)
    conv5 = Conv2D(64, 3, activation='relu', padding='same')(conv5)

    # Выходной слой свертки с функцией активации Softmax под число классов
    output = Conv2D(class_count, 1, activation='softmax', padding='same')(conv5)

    model = Model(inputs=inputs, outputs=output)
    return model

# Создание экземпляра модели
model = unet_model(CLASS_COUNT, (IMG_WIDTH, IMG_HEIGHT, 3))
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 192, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_11 (Conv2D)  │ (None, 192, 256,  │      1,792 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_12 (Conv2D)  │ (None, 192, 256,  │     36,928 │ conv2d_11[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 96, 128,   │          0 │ conv2d_12[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_13 (Conv2D)  │ (None, 96, 128,   │     73,856 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 96, 128,   │    147,584 │ conv2d_13[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 48, 64,    │          0 │ conv2d_14[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 48, 64,    │    295,168 │ max_pooling2d_3[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 48, 64,    │    590,080 │ conv2d_15[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_2  │ (None, 96, 128,   │    131,200 │ conv2d_16[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 96, 128,   │          0 │ conv2d_14[0][0],  │
│ (Concatenate)       │ 256)              │            │ conv2d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 96, 128,   │    295,040 │ concatenate_2[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 96, 128,   │    147,584 │ conv2d_17[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_3  │ (None, 192, 256,  │     32,832 │ conv2d_18[0][0]   │
│ (Conv2DTranspose)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 192, 256,  │          0 │ conv2d_12[0][0],  │
│ (Concatenate)       │ 128)              │            │ conv2d_transpose… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 192, 256,  │     73,792 │ concatenate_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 192, 256,  │     36,928 │ conv2d_19[0][0] 

 Total params: 1,863,824 (7.11 MB)

 Trainable params: 1,863,824 (7.11 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(x_train, y_train,
                    epochs=20,
                    batch_size=16,
                    validation_data=(x_val, y_val))

Epoch 1/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 137s 803ms/step - accuracy: 0.5048 - loss: 1.6359 - val_accuracy: 0.5389 - val_loss: 1.2170
Epoch 2/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 420ms/step - accuracy: 0.5383 - loss: 1.2098 - val_accuracy: 0.5521 - val_loss: 1.1573
Epoch 3/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 49s 409ms/step - accuracy: 0.5616 - loss: 1.1053 - val_accuracy: 0.5725 - val_loss: 1.0914
Epoch 4/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 418ms/step - accuracy: 0.5758 - loss: 1.0644 - val_accuracy: 0.5811 - val_loss: 1.0406
Epoch 5/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 49s 413ms/step - accuracy: 0.5869 - loss: 1.0149 - val_accuracy: 0.5941 - val_loss: 1.0223
Epoch 6/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 418ms/step - accuracy: 0.6059 - loss: 0.9800 - val_accuracy: 0.5541 - val_loss: 1.0391
Epoch 7/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 419ms/step - accuracy: 0.6218 - loss: 0.9444 - val_accuracy: 0.5885 - val_loss: 1.0007
Epoch 8/20
119/119 ━━━━━━━━━━━━━━━━━━━━ 50s 420ms/step - accuracy: 0.6373 - loss: 